# Sports data explorer (Jupyter)

Free-form, **read-only** exploration of the local sports databases.

- `sports_db.q(sql, db)` runs SQL and returns a pandas DataFrame.
- `sports_db.DATABASES` lists what's available; `tables(db)` / `schema(db, table)` browse structure.
- Connections are opened `mode=ro`, so nothing you run here can change a database.
- The full column map is in [`../SCHEMA.md`](../SCHEMA.md) (with exact row counts).

Run the setup cell first, then explore in the scratch cells at the bottom.

In [ ]:
# Setup. Re-imports sports_db automatically when you edit it.
%load_ext autoreload
%autoreload 2

import pandas as pd
import sports_db
from sports_db import q, tables, schema, DATABASES, DATA_ROOT

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print(f"data root: {DATA_ROOT}\n")
for name, path in DATABASES.items():
    mb = path.stat().st_size / 1e6
    print(f"  {name:16} {mb:8.0f} MB")

## Browse structure

`tables(db)` is fast (names only). `schema(db, table)` shows columns + types. For row
counts, see `SCHEMA.md` or run a `COUNT(*)` yourself.

In [ ]:
print("nba tables:", tables("nba"))
schema("nba", "player_game")

## A first query (and why you validate it)

Repo convention: before trusting a derived number, check it against a fact you already
know. The all-time regular-season scoring list should read **LeBron, Kareem, Malone** —
if it doesn't, suspect the query or the data, not reality.

In [ ]:
q("""
    SELECT pl.player_name, SUM(pg.pts) AS career_pts, COUNT(*) AS games
    FROM player_game pg
    JOIN players pl ON pl.player_id = pg.player_id
    WHERE pg.season_type = 'Regular Season'
    GROUP BY pg.player_id
    ORDER BY career_pts DESC
    LIMIT 5
""", "nba")

## 🧩 Your turn: a safety guard

`play_by_play` is **17.7M rows**. `q("SELECT * FROM play_by_play")` would try to pull all
of them into memory. Design a `peek()` that protects against this, then move it into
`sports_db.py` once you like it.

Decisions to make (there's no single right answer):
- Auto-append `LIMIT` when the query has none — simple, but it can **silently truncate**.
- Only guard row-returning `SELECT`s, leave `COUNT`/`SUM`/`GROUP BY` alone — smarter.
- Hard-cap fetched rows and **warn loudly** when the cap is hit — never silent.

Fill in the body below.

In [ ]:
def peek(sql: str, db: str = "nba", limit: int = 1_000) -> pd.DataFrame:
    """Like q(), but guard against accidentally pulling millions of rows."""
    # TODO: your guard here. Then call q(...) to actually run it.
    raise NotImplementedError

# Once implemented, this should NOT melt your laptop:
# peek("SELECT * FROM play_by_play", "nba")

## Scratch space — explore freely

Some starting points: NFL EPA leaders (`nfl` → `player_game.passing_epa`), PGA
strokes-gained (`pga`), biggest NBA comebacks (`nba_comebacks` → `team_game_deficits`).